In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas numpy scikit-learn tabulate

Mounted at /content/drive


In [2]:
import os

# ===== CHINH DUONG DAN O DAY =====
BASE_PATH  = "/content/drive/MyDrive/ĐATN/data"

# Thu muc output cua notebook Ground_Truth
GT_DIR     = f"{BASE_PATH}/processed/ground_truth_text"

# Thu muc chua vector van ban da trich lai
VECTOR_DIR = f"{BASE_PATH}/processed/embeddings/text_features"

# File ket qua se luu
RESULT_CSV = f"{BASE_PATH}/processed/evaluation_text_only_standard.csv"

# Ten mo hinh -> ten file .npy trong VECTOR_DIR
MODELS = {
    "TF-IDF Baseline": "tfidf_embeddings.npy",
    "FastText Baseline": "fasttext_embeddings.npy",
    "SBERT (NLP SOTA)": "sbert_embeddings.npy",
    "Fashion-CLIP (Text-only)": "fashionclip_text_embeddings.npy",
}

K_VALUES = [1, 5, 10]
# =================================

for p in (GT_DIR, VECTOR_DIR):
    print(p, "| ton tai:", os.path.exists(p))

/content/drive/MyDrive/ĐATN/data/processed/ground_truth_text | ton tai: True
/content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features | ton tai: True


In [4]:
import json
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

mapping = pd.read_csv(f"{GT_DIR}/mapping_full.csv")
queries = pd.read_csv(f"{GT_DIR}/queries.csv")
gallery = pd.read_csv(f"{GT_DIR}/gallery.csv")
with open(f"{GT_DIR}/qrels.json", encoding="utf-8") as f:
    qrels = json.load(f)

# Thu tu dong vector = thu tu dong mapping_full.csv sap theo row_in_captions
mapping = mapping.sort_values("row_in_captions").reset_index(drop=True)
mapping["vec_row"] = np.arange(len(mapping))
row_of = dict(zip(mapping["image_id"], mapping["vec_row"]))

query_rows = queries["image_id"].map(row_of)
gallery_rows = gallery["image_id"].map(row_of)
assert not query_rows.isna().any() and not gallery_rows.isna().any(), \
    "Co anh query/gallery khong co trong mapping_full.csv"

query_indices = query_rows.astype(int).values
gallery_indices = gallery_rows.astype(int).values
query_labels = queries["item_id"].to_numpy(dtype=object)
gallery_labels = gallery["item_id"].to_numpy(dtype=object)

# Kiem tra nhat quan giua qrels.json va nhan item_id
assert set(queries["image_id"]).isdisjoint(set(gallery["image_id"])), "Query va gallery giao nhau"
gal_count = pd.Series(gallery_labels).value_counts()
n_rel_all = np.array([gal_count.get(i, 0) for i in query_labels])
assert (n_rel_all >= 1).all(), "Co query khong co anh lien quan trong gallery"
assert all(len(qrels[q]) == n for q, n in zip(queries["image_id"], n_rel_all)), \
    "qrels.json khong khop voi queries.csv/gallery.csv"

print("So anh trong mapping (so dong vector can co):", len(mapping))
print("Query hop le :", len(query_indices))
print("Gallery      :", len(gallery_indices), "anh,", len(set(gallery_labels)), "san pham")
print("Anh lien quan moi query: trung binh %.2f, trung vi %d, toi da %d"
      % (n_rel_all.mean(), np.median(n_rel_all), n_rel_all.max()))

So anh trong mapping (so dong vector can co): 12694
Query hop le : 1893
Gallery      : 2943 anh, 2046 san pham
Anh lien quan moi query: trung binh 2.48, trung vi 2, toi da 15


In [5]:
def evaluate_retrieval(query_vectors, gallery_vectors, q_labels, g_labels, k_list=(1, 5, 10)):
    q_labels = np.asarray(q_labels, dtype=object)
    g_labels = np.asarray(g_labels, dtype=object)
    q = normalize(query_vectors, axis=1)
    g = normalize(gallery_vectors, axis=1)
    sim = q @ g.T                                   # (n_query, n_gallery), cosine

    ng = sim.shape[1]
    k_list = [k for k in k_list if k <= ng]
    max_k = max(k_list)

    # sap xep giam dan, on dinh (tie -> giu thu tu gallery) de ket qua tai lap
    order = np.argsort(-sim, axis=1, kind="stable")[:, :max_k]
    rel = (g_labels[order] == q_labels[:, None]).astype(float)       # (n_query, max_k)

    counts = pd.Series(g_labels).value_counts()
    n_rel = np.array([counts.get(x, 0) for x in q_labels])
    assert (n_rel > 0).all(), "Co query khong co anh lien quan"

    discount = 1.0 / np.log2(np.arange(2, max_k + 2))
    ideal_cum = np.cumsum(discount)

    per_query = {}
    for k in k_list:
        hits = rel[:, :k].sum(axis=1)
        dcg = (rel[:, :k] * discount[:k]).sum(axis=1)
        idcg = ideal_cum[np.minimum(n_rel, k) - 1]

        # Chi giu lai cac do do: Precision, Recall, NDCG
        per_query[f"P@{k}"] = hits / k
        per_query[f"R@{k}"] = hits / n_rel
        per_query[f"NDCG@{k}"] = dcg / idcg

    results = {name: float(v.mean()) for name, v in per_query.items()}
    return results, per_query

print("Da khoi tao ham evaluate_retrieval")

Da khoi tao ham evaluate_retrieval


In [6]:
# 1) Oracle: xep dung tuyet doi -> Recall, NDCG = 1 khi K >= so anh dung
g_lab = np.array(["a", "a", "a", "b", "b", "c"])
q_lab = np.array(["a", "b", "c"])
onehot = {"a": [1.0, 0.0, 0.0], "b": [0.0, 1.0, 0.0], "c": [0.0, 0.0, 1.0]}
G = np.array([onehot[x] for x in g_lab])
Q = np.array([onehot[x] for x in q_lab])
r, _ = evaluate_retrieval(Q, G, q_lab, g_lab, (3,))
assert abs(r["R@3"] - 1) < 1e-9 and abs(r["NDCG@3"] - 1) < 1e-9

# 2) Vi du tinh tay: n_rel = 2, anh dung o hang 1 va 3 (5 anh gallery, cosine giam dan theo hang)
g_lab = np.array(["x", "y", "x", "z", "w"])
q_lab = np.array(["x"])
G = np.array([[1.0, 0.0], [0.9, 0.436], [0.8, 0.6], [0.6, 0.8], [0.3, 0.954]])
Q = np.array([[1.0, 0.0]])
r, _ = evaluate_retrieval(Q, G, q_lab, g_lab, (1, 3))
expected_ndcg3 = (1 + 1 / np.log2(4)) / (1 + 1 / np.log2(3))
assert abs(r["R@1"] - 0.5) < 1e-9
assert abs(r["P@3"] - 2 / 3) < 1e-9 and abs(r["R@3"] - 1.0) < 1e-9
assert abs(r["NDCG@3"] - expected_ndcg3) < 1e-9
print("Kiem tra ham metrics: dat")

Kiem tra ham metrics: dat


In [7]:
evaluation_results = []
per_query_store = {}

def run_one(model_name, all_vectors):
    q_vecs = all_vectors[query_indices]
    g_vecs = all_vectors[gallery_indices]
    res, per_q = evaluate_retrieval(q_vecs, g_vecs, query_labels, gallery_labels, K_VALUES)
    res["Model"] = model_name
    evaluation_results.append(res)
    per_query_store[model_name] = per_q
    return res

print("--- BAT DAU DANH GIA ---")
for model_name, file_name in MODELS.items():
    file_path = f"{VECTOR_DIR}/{file_name}"
    print(f"Dang xu ly: {model_name}...")
    if not os.path.exists(file_path):
        print(f"  Khong tim thay file: {file_path}")
        continue

    all_vectors = np.load(file_path)
    if all_vectors.ndim != 2 or all_vectors.shape[0] != len(mapping):
        print(f"  Sai so dong: file co {all_vectors.shape}, can {len(mapping)} dong (thu tu theo mapping_full.csv).")
        if all_vectors.shape[0] == 12278:
            print("  Day co the la vector cu. Can trich lai vector cho captions_12k.")
        continue
    if not np.isfinite(all_vectors).all():
        print("  Vector co gia tri NaN/Inf, bo qua.")
        continue

    t0 = time.time()
    res = run_one(model_name, all_vectors)
    print(f"  Xong ({time.time() - t0:.2f}s). R@10: {res['R@10']:.4f}, NDCG@10: {res['NDCG@10']:.4f}")

# Moc tham chieu: vector ngau nhien
rng = np.random.default_rng(0)
run_one("Random (moc tham chieu)", rng.normal(size=(len(mapping), 64)))
print("Da them moc tham chieu ngau nhien")

--- BAT DAU DANH GIA ---
Dang xu ly: TF-IDF Baseline...
  Xong (0.89s). R@10: 0.1708, NDCG@10: 0.1166
Dang xu ly: FastText Baseline...
  Xong (0.65s). R@10: 0.0551, NDCG@10: 0.0361
Dang xu ly: SBERT (NLP SOTA)...
  Xong (0.66s). R@10: 0.1366, NDCG@10: 0.0903
Dang xu ly: Fashion-CLIP (Text-only)...
  Xong (0.68s). R@10: 0.4748, NDCG@10: 0.3627
Da them moc tham chieu ngau nhien


In [8]:
df_results = pd.DataFrame(evaluation_results)

cols = ["Model"]
for k in K_VALUES:
    cols.extend([f"P@{k}", f"R@{k}", f"NDCG@{k}"])
cols = [c for c in cols if c in df_results.columns]

print("=" * 100)
print(f"KET QUA TEXT-TO-TEXT ({len(query_indices)} query, {len(gallery_indices)} gallery, anh dung = cung item_id)")
print("=" * 100)
print(df_results[cols].round(4).to_markdown(index=False))

df_results[cols].to_csv(RESULT_CSV, index=False)
print("\nDa luu bang ket qua tai:", RESULT_CSV)

KET QUA TEXT-TO-TEXT (1893 query, 2943 gallery, anh dung = cung item_id)
| Model                    |    P@1 |    R@1 |   NDCG@1 |    P@5 |    R@5 |   NDCG@5 |   P@10 |   R@10 |   NDCG@10 |
|:-------------------------|-------:|-------:|---------:|-------:|-------:|---------:|-------:|-------:|----------:|
| TF-IDF Baseline          | 0.0829 | 0.044  |   0.0829 | 0.047  | 0.1205 |   0.0991 | 0.0341 | 0.1708 |    0.1166 |
| FastText Baseline        | 0.0254 | 0.0089 |   0.0254 | 0.0173 | 0.0367 |   0.0303 | 0.0125 | 0.0551 |    0.0361 |
| SBERT (NLP SOTA)         | 0.065  | 0.0319 |   0.065  | 0.037  | 0.0891 |   0.074  | 0.0282 | 0.1366 |    0.0903 |
| Fashion-CLIP (Text-only) | 0.2895 | 0.1693 |   0.2895 | 0.1447 | 0.3756 |   0.3275 | 0.0966 | 0.4748 |    0.3627 |
| Random (moc tham chieu)  | 0.0011 | 0.0004 |   0.0011 | 0.0011 | 0.0021 |   0.0016 | 0.001  | 0.0032 |    0.0021 |

Da luu bang ket qua tai: /content/drive/MyDrive/ĐATN/data/processed/evaluation_text_only_standard.csv


In [9]:
K_MAIN = 10 if 10 in K_VALUES else max(K_VALUES)
view_arr = queries["view"].to_numpy(dtype=object)

rows = []
for model_name, per_q in per_query_store.items():
    for v in sorted(set(view_arr)):
        mask = view_arr == v
        if mask.sum() < 30:            # bo nhom qua it query
            continue
        rows.append({"Model": model_name, "Goc chup query": v, "So query": int(mask.sum()),
                     f"P@{K_MAIN}": per_q[f"P@{K_MAIN}"][mask].mean(),
                     f"R@{K_MAIN}": per_q[f"R@{K_MAIN}"][mask].mean(),
                     f"NDCG@{K_MAIN}": per_q[f"NDCG@{K_MAIN}"][mask].mean()})

df_view = pd.DataFrame(rows)
print(f"\nPhan tich chuyen sau theo goc chup anh (Tinh voi K={K_MAIN}):\n")
print(df_view.round(4).to_markdown(index=False))


Phan tich chuyen sau theo goc chup anh (Tinh voi K=10):

| Model                    | Goc chup query   |   So query |   P@10 |   R@10 |   NDCG@10 |
|:-------------------------|:-----------------|-----------:|-------:|-------:|----------:|
| TF-IDF Baseline          | additional       |        551 | 0.0341 | 0.1825 |    0.1199 |
| TF-IDF Baseline          | back             |        100 | 0.03   | 0.1502 |    0.1064 |
| TF-IDF Baseline          | front            |        395 | 0.0339 | 0.1896 |    0.1276 |
| TF-IDF Baseline          | full             |        693 | 0.033  | 0.1551 |    0.1058 |
| TF-IDF Baseline          | side             |        151 | 0.0404 | 0.1658 |    0.1317 |
| FastText Baseline        | additional       |        551 | 0.0136 | 0.0601 |    0.039  |
| FastText Baseline        | back             |        100 | 0.01   | 0.034  |    0.0287 |
| FastText Baseline        | front            |        395 | 0.0084 | 0.0568 |    0.0335 |
| FastText Baseline        | ful